# Tangerine 2D — 감귤 병해 5클래스 분류 학습

**논리적 흐름**
1. Colab 경로 설정 (`Tangerine_2D` 데이터, `Tangerine_2D_AI` 코드)
2. 패키지 설치 (`requirements-notebook.txt`)
3. 설정(`TrainConfig`) — 분할 비율, 에폭, 백본 등
4. **`run_training(..., run_final_eval=False)`** — train/val만 → 끝나면 **`training_summary.json`** 저장 (체크포인트·TensorBoard)
5. **`run_test_only(cfg)`** — `best.pt` 로드 후 테스트 리포트·`test_metrics.json`·`test_predictions.csv`·혼동행렬 PNG·**Grad-CAM (`gradcam/`)** — **§4만 돌리면 XAI는 없음. 반드시 이 셀 실행**
6. **`zip_run_directory`** — 실험 폴더 전체를 zip (Colab 다운로드 / Drive 백업)
7. (선택) TensorBoard 실행

데이터는 **클래스별 하위 폴더**만 있으면 됩니다. train/val/test는 코드가 **stratified**로 나눕니다.

자세한 기술 판단·근거는 이 폴더의 `README.md`를 참고하세요.

## §1. 압축 해제 + 경로 자동 설정 (이 셀을 맨 먼저 실행)

- `/content` 에 `Tangerine_2D.zip`, `Tangerine_2D_AI.zip` 이 있으면 **자동으로 풉니다.**  
- 이미 폴더로 넣었으면 zip 없이도 동작합니다.  
- **`Tangerine_2D_AI` 안에 또 `Tangerine_2D_AI` 가 있어도** `config.py` 기준으로 잡습니다.

In [ ]:
import sys
import zipfile
from pathlib import Path

CONTENT = Path("/content")

for zip_name in ("Tangerine_2D.zip", "Tangerine_2D_AI.zip"):
    zp = CONTENT / zip_name
    if zp.is_file():
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(CONTENT)
        print("압축 해제 완료:", zip_name)
    else:
        print("건너뜀 (없음):", zip_name)

_cfg = list(CONTENT.glob("**/Tangerine_2D_AI/config.py"))
if not _cfg:
    raise FileNotFoundError(
        "Tangerine_2D_AI/config.py 를 찾을 수 없습니다.\n"
        "- Tangerine_2D_AI.zip 을 /content 에 업로드한 뒤 이 셀을 다시 실행하거나,\n"
        "- 이미 푼 폴더가 /content 아래 어딘가에 있는지 확인하세요."
    )

PROJECT_ROOT = min((p.parent for p in _cfg), key=lambda p: len(p.parts))
sys.path.insert(0, str(PROJECT_ROOT))

from colab_paths import resolve_data_root

DATA_ROOT = resolve_data_root(CONTENT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)

## §2. 패키지 설치

In [ ]:
import subprocess

req = PROJECT_ROOT / "requirements-notebook.txt"
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)]
)

## §3. 하이퍼파라미터

- `output_dir`: 결과·체크포인트 저장 위치  
- `best_metric`: `val_accuracy` 또는 `val_f1_macro`

In [ ]:
from config import TrainConfig

cfg = TrainConfig(
    data_root=DATA_ROOT,
    output_dir=Path("/content/Tangerine_2D_runs"),
    experiment_name="tangerine_2d",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    backbone="resnet18",
    batch_size=32,
    num_epochs=30,
    learning_rate=1e-4,
    num_workers=2,
    seed=42,
    freeze_backbone_epochs=0,
    xai_num_samples=12,
    best_metric="val_f1_macro",
)
cfg.validate_split()
print(cfg)

## §4. 학습만 (train / validation)

`best.pt`, `last.pt`, TensorBoard, **`training_summary.json`** (베스트 에폭·val 지표). 테스트·Grad-CAM은 **하지 않음** — **다음 섹션 §5 필수**.

In [ ]:
from experiment import run_training

out_dir = run_training(cfg, run_final_eval=False)
print("출력 디렉터리:", out_dir)

## §5. 테스트 · Grad-CAM · 결과 zip

`best.pt` 로드 후 리포트, `test_metrics.json`, `test_predictions.csv`, 혼동 행렬 PNG, **Grad-CAM**, 이어서 실험 폴더 **zip** (학습과 분리).

In [ ]:
from experiment import run_test_only, zip_run_directory

out_test = run_test_only(cfg)
zip_path = zip_run_directory(out_test)
print("실험 폴더:", out_test)
print("Zip 경로:", zip_path)

## §5b (선택) zip만 만들기

이미 §5를 실행해 두었거나, 로컬에서 `best.pt`만 있는 폴더를 묶어 백업할 때. `cfg`가 위에서 정의되어 있어야 합니다.

In [ ]:
from pathlib import Path
from experiment import zip_run_directory

out_dir = cfg.effective_output_dir()
print("압축:", zip_run_directory(out_dir))
# Google Drive에 직접 쓰려면:
# zip_run_directory(out_dir, Path("/content/drive/MyDrive/tangerine_2d_bundle.zip"))

## §6. TensorBoard (선택)

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir /content/Tangerine_2D_runs/tangerine_2d/tensorboard

## §7. 다른 경로의 `best.pt` (선택)

Drive에만 체크포인트가 있을 때 주석 해제.

In [ ]:
# from pathlib import Path
# from experiment import run_test_only
# run_test_only(cfg, Path("/content/drive/MyDrive/best.pt"))